# ⚔️ 17. Feature Engineering: RGB + AVG + NTSC Training SVM (Multi-Domain Preprocessing & Validation Selection)

Notebook ini melatih Support Vector Machine (SVM) pada **Combined Feature Vector 1.536-dimensi** hasil Feature Fusion Tahap 16 dengan preprocessing dan hyperparameter yang dapat dikonfigurasi secara modular:
- **Anti Data Leakage**: Seluruh preprocessor (StandardScaler, L2 Normalizer, Weighting) hanya di-fit pada data training.
- **Strategi Preprocessing Terkonfigurasi**:
  - Mode A: `StandardScaler` global
  - Mode B: `StandardScaler` + `L2 Normalizer` (Hyperspherical Cosine Space)
  - Mode C: `Per-domain StandardScaler` (menstandarisasi masing-masing domain 512-D secara independen)
  - Mode D: `Limited Feature Weighting` (memberikan bobot adaptif antar domain: RGB, AVG, NTSC)
- **Hierarchical Specialist Classifier (Optional)**: Menyiapkan classifier bertingkat untuk pasangan kelas ambigu ({cat, dog} dan {automobile, truck}) yang dievaluasi murni pada data validasi.
- **Pemilihan Parameter**: Seluruh tuning dan perbandingan dilakukan terhadap partisi **45.000 train / 5.000 validation**, tanpa melihat data testing.


In [1]:
import os
import sys
import pickle
import json
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.svm import SVC

ARTIFACT_DIR = 'cifar10_artifacts/rgb_avg_ntsc'
train_feat_path = os.path.join(ARTIFACT_DIR, 'train_features.npz')
print(f"Memuat vektor fitur gabungan training dari: {train_feat_path}")
train_data = np.load(train_feat_path)
X_train_features = train_data['X_train_features']
y_train = train_data['y_train']
print(f"Dimensi fitur gabungan: {X_train_features.shape}")

RANDOM_STATE = 23092026


Memuat vektor fitur gabungan training dari: cifar10_artifacts/rgb_avg_ntsc\train_features.npz
Dimensi fitur gabungan: (50000, 1536)


## 🛠️ 1. StandardScaler + L2-Hyperspherical Normalization & Melatih Model SVM Classifier (C=25.0)


In [2]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# ==============================================================================
# 1. Custom Transformer: Multi-Domain Feature Preprocessor (1.536-Dimensi)
# ==============================================================================
class FusedDomainPreprocessor(BaseEstimator, TransformerMixin):
    """
    Preprocessor modular untuk fitur gabungan 1.536 dimensi (RGB: 512, AVG: 512, NTSC: 512).
    Mendukung 4 strategi:
      Mode A: 'standard'           -> StandardScaler global (tanpa L2)
      Mode B: 'standard_l2'        -> StandardScaler global + L2 Normalizer
      Mode C: 'per_domain_standard'-> StandardScaler independen per domain + L2 Normalizer
      Mode D: 'domain_weighted'    -> StandardScaler per domain + pembobotan (w_rgb, w_avg, w_ntsc) + optional L2
    """
    def __init__(self, mode='domain_weighted', weights=(1.0, 1.0, 1.0), apply_l2=True):
        self.mode = mode
        self.weights = weights
        self.apply_l2 = apply_l2
        self.scaler_rgb = None
        self.scaler_avg = None
        self.scaler_ntsc = None
        self.scaler_global = None
        self.normalizer = Normalizer(norm='l2') if apply_l2 else None

    def fit(self, X, y=None):
        if self.mode in ['per_domain_standard', 'domain_weighted']:
            self.scaler_rgb = StandardScaler().fit(X[:, :512])
            self.scaler_avg = StandardScaler().fit(X[:, 512:1024])
            self.scaler_ntsc = StandardScaler().fit(X[:, 1024:1536])
        else:
            self.scaler_global = StandardScaler().fit(X)
        return self

    def transform(self, X):
        w_rgb, w_avg, w_ntsc = self.weights
        if self.scaler_global is not None:
            X_scaled = self.scaler_global.transform(X)
            if self.mode == 'domain_weighted':
                X_scaled[:, :512] *= w_rgb
                X_scaled[:, 512:1024] *= w_avg
                X_scaled[:, 1024:1536] *= w_ntsc
            out = X_scaled
        else:
            x_rgb = self.scaler_rgb.transform(X[:, :512]) * w_rgb
            x_avg = self.scaler_avg.transform(X[:, 512:1024]) * w_avg
            x_ntsc = self.scaler_ntsc.transform(X[:, 1024:1536]) * w_ntsc
            out = np.hstack([x_rgb, x_avg, x_ntsc])

        if self.apply_l2 and self.normalizer is not None:
            out = self.normalizer.transform(out)
        return out

# ==============================================================================
# 2. Hierarchical Specialist Classifier (Optional Architecture)
# ==============================================================================
class HierarchicalSpecialistClassifier(BaseEstimator, ClassifierMixin):
    """
    Hierarchical Classifier untuk CIFAR-10:
    - Base classifier: Multiclass SVM pada ruang 1.536 dimensi
    - Specialist 1: Binary SVM khusus membedakan Cat (3) vs Dog (5)
    - Specialist 2: Binary SVM khusus membedakan Automobile (1) vs Truck (9)
    """
    def __init__(self, base_model, specialist_cat_dog=None, specialist_auto_truck=None, use_specialist=False):
        self.base_model = base_model
        self.specialist_cat_dog = specialist_cat_dog
        self.specialist_auto_truck = specialist_auto_truck
        self.use_specialist = use_specialist

    def fit_specialists(self, X_scaled, y):
        # 1. Specialist cat (3) vs dog (5)
        mask_cd = np.isin(y, [3, 5])
        if np.sum(mask_cd) > 0:
            print(f"Melatih Specialist Cat-vs-Dog pada {np.sum(mask_cd)} sampel data training...")
            self.specialist_cat_dog = SVC(C=10.0, gamma='scale', kernel='rbf', random_state=RANDOM_STATE)
            self.specialist_cat_dog.fit(X_scaled[mask_cd], y[mask_cd])

        # 2. Specialist automobile (1) vs truck (9)
        mask_at = np.isin(y, [1, 9])
        if np.sum(mask_at) > 0:
            print(f"Melatih Specialist Auto-vs-Truck pada {np.sum(mask_at)} sampel data training...")
            self.specialist_auto_truck = SVC(C=10.0, gamma='scale', kernel='rbf', random_state=RANDOM_STATE)
            self.specialist_auto_truck.fit(X_scaled[mask_at], y[mask_at])
        return self

    def predict(self, X_scaled):
        preds = self.base_model.predict(X_scaled)
        if not self.use_specialist:
            return preds

        preds = preds.copy()
        if self.specialist_cat_dog is not None:
            cd_idx = np.where(np.isin(preds, [3, 5]))[0]
            if len(cd_idx) > 0:
                preds[cd_idx] = self.specialist_cat_dog.predict(X_scaled[cd_idx])

        if self.specialist_auto_truck is not None:
            at_idx = np.where(np.isin(preds, [1, 9]))[0]
            if len(at_idx) > 0:
                preds[at_idx] = self.specialist_auto_truck.predict(X_scaled[at_idx])

        return preds

    def decision_function(self, X_scaled):
        return self.base_model.decision_function(X_scaled)

# ==============================================================================
# 3. Validation-Driven Configuration Selection (Disjoint dari Validation CNN)
# ==============================================================================
# Partisi validation dibuat DISJOINT dari 5.000 sampel validation set CNN (zero overlap)
indices_all = np.arange(len(X_train_features))
idx_cnn_train, idx_cnn_val = train_test_split(
    indices_all,
    test_size=0.10,
    random_state=RANDOM_STATE,
    stratify=y_train
)

idx_svm_subtrain, idx_svm_val = train_test_split(
    idx_cnn_train,
    test_size=5000,
    random_state=RANDOM_STATE + 100,
    stratify=y_train[idx_cnn_train]
)

X_tr_sub, y_tr_sub = X_train_features[idx_svm_subtrain], y_train[idx_svm_subtrain]  # 40.000 sampel
X_va_sub, y_va_sub = X_train_features[idx_svm_val], y_train[idx_svm_val]            # 5.000 sampel disjoint

print("=" * 76)
print("  VALIDATION-DRIVEN SELECTION (40k Sub-Train / 5k Disjoint SVM Validation)")
print("  Mengevaluasi seluruh kandidat Preprocessing, Feature Weights, L2, dan C")
print("=" * 76)
print(f"Data Sub-Train SVM : {X_tr_sub.shape}")
print(f"Data Val SVM       : {X_va_sub.shape} (Disjoint dari CNN Validation)\n")

# Daftar seluruh kandidat Preprocessing & Feature Weighting
# Tidak ada yang diasumsikan optimal secara apriori; semuanya diuji pada validation set!
preprocessing_candidates = [
    ('Mode A : StandardScaler Global (No L2)',    'standard',            (1.0, 1.0, 1.0), False),
    ('Mode B : StandardScaler Global + L2',       'standard_l2',         (1.0, 1.0, 1.0), True),
    ('Mode C : Per-Domain StandardScaler + L2',   'per_domain_standard', (1.0, 1.0, 1.0), True),
    ('Mode D1: Weighted (1.0, 1.0, 1.0) + L2',    'domain_weighted',     (1.0, 1.0, 1.0), True),
    ('Mode D2: Weighted (1.0, 0.9, 0.9) + L2',    'domain_weighted',     (1.0, 0.9, 0.9), True),
    ('Mode D3: Weighted (1.0, 0.8, 0.8) + L2',    'domain_weighted',     (1.0, 0.8, 0.8), True),
    ('Mode D4: Weighted (1.0, 0.7, 0.7) + L2',    'domain_weighted',     (1.0, 0.7, 0.7), True),
    ('Mode D5: Weighted (1.0, 0.6, 0.6) + L2',    'domain_weighted',     (1.0, 0.6, 0.6), True),
    ('Mode D6: Weighted (1.1, 0.9, 0.9) + L2',    'domain_weighted',     (1.1, 0.9, 0.9), True),
    ('Mode D7: Weighted (1.1, 1.0, 0.9) + L2',    'domain_weighted',     (1.1, 1.0, 0.9), True),
    ('Mode D8: Weighted (1.0, 0.8, 0.8) (No L2)', 'domain_weighted',     (1.0, 0.8, 0.8), False),
]

SVM_GAMMA = 'scale'
KERNEL = 'rbf'

best_val_score = -1.0
best_config = {}

# --- Tahap 1: Evaluasi Kandidat Preprocessing & Feature Weighting ---
print("--- Tahap 1: Evaluasi Kandidat Preprocessing & Feature Weighting (Baseline C=20.0) ---")
for name, mode, w, l2_flag in preprocessing_candidates:
    prep = FusedDomainPreprocessor(mode=mode, weights=w, apply_l2=l2_flag)
    X_tr_prep = prep.fit_transform(X_tr_sub)
    X_va_prep = prep.transform(X_va_sub)
    
    clf = SVC(C=20.0, gamma=SVM_GAMMA, kernel=KERNEL, cache_size=2048, random_state=RANDOM_STATE)
    clf.fit(X_tr_prep, y_tr_sub)
    score = clf.score(X_va_prep, y_va_sub)
    print(f"  {name:44s} -> Disjoint Val Acc: {score*100:.2f}%")
    
    if score > best_val_score:
        best_val_score = score
        best_config = {
            'mode': mode,
            'weights': w,
            'apply_l2': l2_flag,
            'C': 20.0,
            'use_specialist': False,
            'score': score
        }

print(f"\n[TERPILIH TAHAP 1] Mode: '{best_config['mode']}', Weights: {best_config['weights']}, L2: {best_config['apply_l2']} (Val Acc: {best_val_score*100:.2f}%)")

# --- Tahap 2: Fine-Tuning Hyperparameter C pada Konfigurasi Preprocessing Terbaik ---
print(f"\n--- Tahap 2: Fine-Tuning C pada Preprocessing Terpilih ---")
best_prep = FusedDomainPreprocessor(
    mode=best_config['mode'],
    weights=best_config['weights'],
    apply_l2=best_config['apply_l2']
)
X_tr_best_prep = best_prep.fit_transform(X_tr_sub)
X_va_best_prep = best_prep.transform(X_va_sub)

C_CANDIDATES = [10.0, 15.0, 20.0, 25.0, 30.0]
for c_cand in C_CANDIDATES:
    clf = SVC(C=c_cand, gamma=SVM_GAMMA, kernel=KERNEL, cache_size=2048, random_state=RANDOM_STATE)
    clf.fit(X_tr_best_prep, y_tr_sub)
    score = clf.score(X_va_best_prep, y_va_sub)
    print(f"  Kandidat C = {c_cand:4.1f} -> Disjoint Val Acc: {score*100:.2f}%")
    
    if score > best_val_score:
        best_val_score = score
        best_config['C'] = c_cand
        best_config['score'] = score

print(f"[TERPILIH TAHAP 2] C Optimal = {best_config['C']} (Val Acc: {best_val_score*100:.2f}%)")

# --- Tahap 3: Evaluasi Hierarchical Specialist Classifier pada Data Validasi ---
print(f"\n--- Tahap 3: Evaluasi Hierarchical Specialist Classifier pada Data Validasi ---")
clf_base = SVC(C=best_config['C'], gamma=SVM_GAMMA, kernel=KERNEL, cache_size=2048, random_state=RANDOM_STATE)
clf_base.fit(X_tr_best_prep, y_tr_sub)

spec_clf = HierarchicalSpecialistClassifier(base_model=clf_base, use_specialist=True)
spec_clf.fit_specialists(X_tr_best_prep, y_tr_sub)
spec_score = accuracy_score(y_va_sub, spec_clf.predict(X_va_best_prep))

print(f"  Base Multiclass SVM (C={best_config['C']})   -> Disjoint Val Acc: {best_val_score*100:.2f}%")
print(f"  With Hierarchical Specialists       -> Disjoint Val Acc: {spec_score*100:.2f}%")

if spec_score > best_val_score:
    best_val_score = spec_score
    best_config['use_specialist'] = True
    print("  [HASIL VALIDASI] Hierarchical Specialist terbukti meningkatkan akurasi -> DIAKTIFKAN")
else:
    best_config['use_specialist'] = False
    print("  [HASIL VALIDASI] Base Multiclass SVM lebih unggul pada validasi -> SPECIALIST DINONAKTIFKAN")

print("\n" + "=" * 76)
print("  KONFIGURASI FINAL TERPILIH MURNI BERDASARKAN HASIL VALIDATION SET:")
print(f"    1. Preprocessing Mode : {best_config['mode']}")
print(f"    2. Feature Weights    : {best_config['weights']}")
print(f"    3. Apply L2           : {best_config['apply_l2']}")
print(f"    4. Hyperparameter C   : {best_config['C']}")
print(f"    5. Use Specialist     : {best_config['use_specialist']}")
print(f"    6. Peak Val Accuracy  : {best_val_score*100:.2f}%")
print("=" * 76)

PREPROCESSING_MODE = best_config['mode']
FEATURE_WEIGHTS    = best_config['weights']
APPLY_L2           = best_config['apply_l2']
SVM_C              = best_config['C']
USE_SPECIALIST     = best_config['use_specialist']

# ==============================================================================
# 4. Pelatihan Model Final pada Seluruh 50.000 Sampel Data Training
# ==============================================================================
print(f"\n4. Menerapkan Preprocessor Final ({PREPROCESSING_MODE}, Weights={FEATURE_WEIGHTS}, L2={APPLY_L2})...")
scaler = FusedDomainPreprocessor(
    mode=PREPROCESSING_MODE,
    weights=FEATURE_WEIGHTS,
    apply_l2=APPLY_L2
)
X_train_scaled = scaler.fit_transform(X_train_features)
print(f"[BERHASIL] Fitur training gabungan dipreprocessing. Shape: {X_train_scaled.shape}")

print(f"\n5. Melatih Base SVM Classifier (C={SVM_C}, gamma='{SVM_GAMMA}', kernel='{KERNEL}')...")
base_svm_model = SVC(
    C=SVM_C,
    gamma=SVM_GAMMA,
    kernel=KERNEL,
    cache_size=4096,
    random_state=RANDOM_STATE,
    decision_function_shape='ovr'
)
base_svm_model.fit(X_train_scaled, y_train)
print("[BERHASIL] Base SVM Classifier telah selesai dilatih!")

if USE_SPECIALIST:
    print("\n[HIERARCHICAL CLASSIFIER] Melatih Specialist Models untuk Cat vs Dog dan Auto vs Truck...")
    svm_model = HierarchicalSpecialistClassifier(base_model=base_svm_model, use_specialist=True)
    svm_model.fit_specialists(X_train_scaled, y_train)
    print("[BERHASIL] Hierarchical Specialist Classifier aktif dan siap digunakan!")
else:
    svm_model = base_svm_model
    print("[INFO] Pipeline menggunakan Base SVM Model murni (Specialist nonaktif).")

sample_acc = accuracy_score(y_train[:5000], svm_model.predict(X_train_scaled[:5000]))
print(f"Akurasi model pada sampel training: {sample_acc * 100:.2f}%")


  VALIDATION-DRIVEN SELECTION (40k Sub-Train / 5k Disjoint SVM Validation)
  Mengevaluasi seluruh kandidat Preprocessing, Feature Weights, L2, dan C
Data Sub-Train SVM : (40000, 1536)
Data Val SVM       : (5000, 1536) (Disjoint dari CNN Validation)

--- Tahap 1: Evaluasi Kandidat Preprocessing & Feature Weighting (Baseline C=20.0) ---
  Mode A : StandardScaler Global (No L2)       -> Disjoint Val Acc: 99.78%
  Mode B : StandardScaler Global + L2          -> Disjoint Val Acc: 99.78%
  Mode C : Per-Domain StandardScaler + L2      -> Disjoint Val Acc: 99.78%
  Mode D1: Weighted (1.0, 1.0, 1.0) + L2       -> Disjoint Val Acc: 99.78%
  Mode D2: Weighted (1.0, 0.9, 0.9) + L2       -> Disjoint Val Acc: 99.80%
  Mode D3: Weighted (1.0, 0.8, 0.8) + L2       -> Disjoint Val Acc: 99.80%
  Mode D4: Weighted (1.0, 0.7, 0.7) + L2       -> Disjoint Val Acc: 99.82%
  Mode D5: Weighted (1.0, 0.6, 0.6) + L2       -> Disjoint Val Acc: 99.80%
  Mode D6: Weighted (1.1, 0.9, 0.9) + L2       -> Disjoint Val 

## 📦 2. Menyimpan Scaler, Model SVM & Metadata ke Disk


In [3]:
scaler_save_path = os.path.join(ARTIFACT_DIR, 'scaler.pkl')
svm_save_path    = os.path.join(ARTIFACT_DIR, 'svm_model.pkl')
meta_save_path   = os.path.join(ARTIFACT_DIR, 'metadata.json')

with open(scaler_save_path, 'wb') as f:
    pickle.dump(scaler, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(svm_save_path, 'wb') as f:
    pickle.dump(svm_model, f, protocol=pickle.HIGHEST_PROTOCOL)

metadata = {
    'method': 'Feature Engineering (RGB + AVG + NTSC) + Scaler + SVM',
    'fused_components': ['RGB (512)', 'Grayscale AVG (512)', 'Grayscale NTSC (512)'],
    'preprocessing_mode': PREPROCESSING_MODE,
    'feature_weights': list(FEATURE_WEIGHTS),
    'apply_l2': APPLY_L2,
    'use_specialist': USE_SPECIALIST,
    'svm_C': float(SVM_C),
    'svm_gamma': str(SVM_GAMMA),
    'total_feature_dim': int(X_train_features.shape[1]),
    'selection_note': 'Configured and selected based on validation set'
}

with open(meta_save_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"[BERHASIL] Scaler tersimpan ke: {scaler_save_path}")
print(f"[BERHASIL] SVM Model tersimpan ke: {svm_save_path}")
print(f"[BERHASIL] Metadata tersimpan ke: {meta_save_path}")


[BERHASIL] Scaler tersimpan ke: cifar10_artifacts/rgb_avg_ntsc\scaler.pkl
[BERHASIL] SVM Model tersimpan ke: cifar10_artifacts/rgb_avg_ntsc\svm_model.pkl
[BERHASIL] Metadata tersimpan ke: cifar10_artifacts/rgb_avg_ntsc\metadata.json
